# 02 · Etiquetas

Formulación convencional vs. censura + landmarking.

In [ ]:
import sys; sys.path.append('..')
import utils

%load_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import utils

def read_ids(split):
    return pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet',
                           columns=[utils.ID, utils.T])

In [ ]:
ro = read_ids('train')
tte = utils.load_raw('train', 'tte')

conv = utils.build_landmarks(ro, tte, k=1)    # convencional: 1 lectura al azar
lm   = utils.build_landmarks(ro, tte, k=10)   # censura + landmarking

utils.to_parquet(conv, 'train_labels_conv')
utils.to_parquet(lm,   'train_labels_landmark')

print('Convencional:', conv.shape)
print(conv['y_conv'].value_counts().sort_index().to_string())
print('\nLandmarking:', lm.shape)
print(lm['y_cens'].value_counts(dropna=False).sort_index().to_string())
print('\nCensurados (%):', round(100 * (1 - tte['in_study_repair'].mean()), 1))

In [ ]:
for split in ['validation', 'test']:
    ev = utils.eval_points(read_ids(split), utils.load_raw(split, 'labels'))
    utils.to_parquet(ev, f'{split}_eval')
    print(split, ev.shape)
    print(ev['class_label'].value_counts().sort_index().to_string(), '\n')